Human in the loop

In [1]:
import os
from dotenv import load_dotenv

In [2]:
from langchain_groq import ChatGroq

c:\Swdtools\conda_envs\py311_agenticai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [4]:
llm.invoke("What is the capital of France?")

AIMessage(content='The capital of France is **Paris**.', additional_kwargs={'reasoning_content': 'The user asks a simple factual question. Answer: Paris.'}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 78, 'total_tokens': 109, 'completion_time': 0.06451331, 'prompt_time': 0.002901568, 'queue_time': 0.049224272, 'total_time': 0.067414878}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_ff6aa7708c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--258be688-121b-4b3d-9ddb-8d909df0b970-0', usage_metadata={'input_tokens': 78, 'output_tokens': 31, 'total_tokens': 109})

In [5]:
from langchain_core.tools import tool

In [6]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [7]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b

In [8]:
multiply({"a":5, "b":6})

C:\Users\prayag sonar\AppData\Local\Temp\ipykernel_23144\2635593141.py:1: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  multiply({"a":5, "b":6})


30

In [9]:
@tool
def search(query: str) -> str:
    """Searches the web for a query and returns the top result."""
    tavily = TavilySearchResults(query=query)
    result = tavily.invoke(query)

    return f"Result for '{query}': {result}"

In [10]:
print(search({"query": "What is the capital of Australia?"}))

C:\Users\prayag sonar\AppData\Local\Temp\ipykernel_23144\3740474683.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily = TavilySearchResults(query=query)


Result for 'What is the capital of Australia?': [{'title': 'What is the capital of Australia? If you thought it was Sydney, keep ...', 'url': 'https://youtooproject.com/en/blog/australia-en/capital-of-australia-canberra/', 'content': 'If you are surprised to learn that this is not the case, maybe you will give Melbourne a try. Well, it’s not either! The capital of Australia is Canberra. Now, don’t just keep the answer for your next trivia game with your friends and discover the whole story.\n\n## The origin of Australia’s capital [...] Skip to navigation\n\nContact us\n\n# What is the capital of Australia? If you thought it was Sydney, keep on reading\n\nIndex\n\nChances are that the first answer that comes to mind when wondering what the capital of Australia is might be wrong. It is all too easy to think that Sydney is the capital of Australia. [...] And this is the brief history of how Canberra became Australia’s capital city. Although, honestly, we must admit that this title is mere

In [11]:
tools = [multiply, search]
tools

[StructuredTool(name='multiply', description='Multiplies two numbers.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x00000138A7E44900>),
 StructuredTool(name='search', description='Searches the web for a query and returns the top result.', args_schema=<class 'langchain_core.utils.pydantic.search'>, func=<function search at 0x00000138A7E47060>)]

In [12]:
llm_with_tools = llm.bind_tools(tools)

In [13]:
result = llm_with_tools.invoke("What is the capital of Australia? Also, what is 7 times 8?")

In [14]:
result.content

'The capital of Australia is **Canberra**.\n\n\\(7 \\times 8 = 56\\).'

In [15]:
tool_mapping = ({tool.name: tool for tool in tools})

In [16]:
tool_mapping['search']

StructuredTool(name='search', description='Searches the web for a query and returns the top result.', args_schema=<class 'langchain_core.utils.pydantic.search'>, func=<function search at 0x00000138A7E47060>)

In [17]:
tool_mapping['search'].invoke({"query":"What is the capital of Australia?"})

'Result for \'What is the capital of Australia?\': [{\'title\': \'What is the capital of Australia? If you thought it was Sydney, keep ...\', \'url\': \'https://youtooproject.com/en/blog/australia-en/capital-of-australia-canberra/\', \'content\': \'If you are surprised to learn that this is not the case, maybe you will give Melbourne a try. Well, it’s not either! The capital of Australia is Canberra. Now, don’t just keep the answer for your next trivia game with your friends and discover the whole story.\\n\\n## The origin of Australia’s capital [...] Skip to navigation\\n\\nContact us\\n\\n# What is the capital of Australia? If you thought it was Sydney, keep on reading\\n\\nIndex\\n\\nChances are that the first answer that comes to mind when wondering what the capital of Australia is might be wrong. It is all too easy to think that Sydney is the capital of Australia. [...] And this is the brief history of how Canberra became Australia’s capital city. Although, honestly, we must admit

In [18]:
tool_mapping[result.tool_calls[0]["name"]].invoke(result.tool_calls[0]["args"])


IndexError: list index out of range

In [ ]:
from typing import TypedDict, Sequence, Annotated
import operator
from langchain_core.messages import BaseMessage

In [ ]:
class AgentState(TypedDict):
    """state of a agent"""
    messages:Annotated[Sequence[BaseMessage], operator.add]

